# Monolith assistant: LoRA fine-tune of LFM2.5-VL-450M
Runtime → Change runtime type → **T4 GPU**. Run top to bottom. This notebook was written against the model card's
Transformers ≥ 5.1 API and has not been run on a GPU yet: if a cell fails, fix it here and tell the repo.

In [ ]:
!pip -q install "transformers>=5.1" peft accelerate pillow
!git clone -q https://github.com/nicosbrooks/Monolith.git -b ui-glass-rework
%cd Monolith/training
!python -m pytest -q tests

## 1. Data
Synthetic, in the app's exact prompt format. The test split uses phrasings and titles training never sees.

In [ ]:
!python generate_data.py --n 6000 --out data --images
import json; rows=[json.loads(l) for l in open('data/train.jsonl')]; print(len(rows)); print(rows[0]['messages'][1]['content']); print(rows[0]['messages'][2]['content'])

## 2. Baseline
The untuned model with the app's ACT_GUIDE. This is the score to beat.

In [ ]:
MODEL='LiquidAI/LFM2.5-VL-450M'
!python evaluate.py --model $MODEL --data data/test.jsonl --guide --limit 200

## 3. Train (LoRA, assistant tokens only)

In [ ]:
import json, random, torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

proc = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL, torch_dtype=torch.float32, device_map={'':0})
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', exclude_modules=r'.*(vision|siglip|projector).*', task_type='CAUSAL_LM'))
model.print_trainable_parameters()

def as_msgs(row, upto):
    msgs = row['messages'][:upto]
    if row.get('image'):
        # the picture belongs to the user turn
        u = [m for m in msgs if m['role']=='user'][-1]
        u['content'] = [{'type':'image'},{'type':'text','text':u['content']}]
    return [m if isinstance(m['content'], list) else {'role':m['role'],'content':[{'type':'text','text':m['content']}]} for m in msgs]

class Data(torch.utils.data.Dataset):
    def __init__(s, path): s.rows=[json.loads(l) for l in open(path)]
    def __len__(s): return len(s.rows)
    def __getitem__(s, i): return s.rows[i]

def collate(batch):
    ids, labels, extra = [], [], {}
    texts, prefixes, imgs = [], [], []
    for r in batch:
        r = json.loads(json.dumps(r))
        prefixes.append(proc.apply_chat_template(as_msgs(r,2), add_generation_prompt=True, tokenize=False))
        full = as_msgs(r,3)
        texts.append(proc.apply_chat_template(full, tokenize=False))
        if r.get('image'): imgs.append(Image.open(r['image']).convert('RGB'))
    enc = proc(text=texts, images=imgs or None, return_tensors='pt', padding=True)
    pre = proc(text=prefixes, images=imgs or None, return_tensors='pt', padding=True)
    labels = enc['input_ids'].clone()
    for i in range(len(batch)):
        n = int(pre['attention_mask'][i].sum())            # prompt length (assumes right padding)
        labels[i,:n] = -100                                 # train on the reply only
    labels[enc['attention_mask']==0] = -100
    enc['labels'] = labels
    return enc

proc.tokenizer.padding_side = 'right'
args = TrainingArguments(output_dir='out', per_device_train_batch_size=4, gradient_accumulation_steps=8, num_train_epochs=2,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.05, logging_steps=10, save_strategy='epoch',
    fp16=False, bf16=False, remove_unused_columns=False, report_to='none', gradient_checkpointing=True)
Trainer(model=model, args=args, train_dataset=Data('data/train.jsonl'), data_collator=collate).train()
model.save_pretrained('out/lora'); proc.save_pretrained('out/lora')

Note: weights are float32 with the LoRA adapters trained in float32 because fp16 training on a T4 needs a loss scaler
the plain Trainer sets up only when `fp16=True`. If memory runs out, lower the batch size to 2 and raise accumulation to 16.

In [ ]:
del model; torch.cuda.empty_cache()
!python evaluate.py --model $MODEL --adapter out/lora --data data/test.jsonl --limit 300

## 4. Export
Merge the adapter, then convert for the platform. Which converter to use depends on where it runs:
- **Web (LiteRT-LM)**: Liquid's `litert-community/LFM2.5-VL-450M` shows the layout; follow Liquid's export notes.
- **Web (ONNX / Transformers.js)** and **iOS (MLX)**: use Liquid's published export tools on the merged folder.
- **GGUF**: `llama.cpp` convert script, for desktop apps.
Always re-run `evaluate.py` on the *merged* weights before converting, and again on the converted file if a runner is available.

In [ ]:
from peft import PeftModel
base = AutoModelForImageTextToText.from_pretrained(MODEL, torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, 'out/lora').merge_and_unload()
merged.save_pretrained('out/merged'); proc.save_pretrained('out/merged')
!ls -lh out/merged